In [ ]:
from pystac_client import Client
import planetary_computer
import odc.stac
import numpy as np
import warnings
import time

warnings.filterwarnings("ignore")

catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=[78.0, 17.0, 78.03, 17.03],
    datetime="2023-01-01/2023-01-10",
    query={"eo:cloud_cover": {"lt": 5}}
)

items = [planetary_computer.sign(item) for item in list(search.get_items())[:2]]

# Use larger chunks to minimize HTTP GET overhead
data = odc.stac.load(
    items,
    bands=["red", "nir"],
    resolution=10,
    crs="EPSG:32644",
    chunks={"x": 2048, "y": 2048} 
)

# Vectorized NDVI and Change
ndvi = (data.nir - data.red) / (data.nir + data.red + 1e-6)
ndvi_diff = (ndvi.isel(time=1) - ndvi.isel(time=0)).fillna(0)

# The "Cloud" Way: Use map_blocks or coarsen to process tiles in parallel
# Coarsen reduces the 512x512 tile to a single mean value per tile lazily
TILE_SIZE = 512
THRESHOLD = 0.15

start = time.time()

# Calculate mean of absolute change per 512x512 block across the whole spatial grid
tile_means = np.abs(ndvi_diff).coarsen(x=TILE_SIZE, y=TILE_SIZE, boundary="pad").mean()

# Trigger a SINGLE compute call for the entire grid of means
computed_means = tile_means.compute()

# Count results
important_tiles = (computed_means > THRESHOLD).sum().item()
total_tiles = computed_means.size
elapsed = time.time() - start

print(f"Total tiles: {total_tiles}")
print(f"Important tiles: {important_tiles}")
print(f"Time: {elapsed:.2f} sec")